In [1]:
# Importações
import pandas as pd
import numpy as np

## Carregamento dos Dados

In [2]:
# Carrega dataframe de pedidos
pedidos_df = pd.read_csv('data/pedidos.csv')

## Análise Exploratória dos Dados (EDA)

In [3]:
# Visão geral dos dados
print(pedidos_df.head())
print(pedidos_df.tail())
print(pedidos_df.info())
print(pedidos_df.describe())
print(pedidos_df.shape)
print(pedidos_df.columns)

   ID_Pedido        Data  Cliente_ID            Item  Quantidade  \
0          1  2023-11-16         222  Torta de Limao        23.0   
1          2  2024-05-25         232          Esfiha         NaN   
2          3  2024-12-20         276  Wrap de Frango         7.0   
3          4  2024-10-17         198    Batata Frita         NaN   
4          5  2023-07-06         385          Esfiha        17.0   

   Preco_Unitario  
0           12.93  
1            5.87  
2           25.98  
3           11.62  
4            6.94  
     ID_Pedido        Data  Cliente_ID             Item  Quantidade  \
445        446  2024-02-27         360    Salada Caesar        30.0   
446        447  2025-05-29         228    Salada Caesar        24.0   
447        448  2025-08-23         243     Agua Mineral        26.0   
448        449  2024-08-30         191  Pizza Calabresa         9.0   
449        450  2025-09-29         122         Yakisoba         8.0   

     Preco_Unitario  
445           27.35  


## Tratamento de Valores Ausentes

In [4]:
# Verifica a quantidade de registros com valores ausentes
pedidos_df.isna().sum()

ID_Pedido          0
Data               0
Cliente_ID         0
Item               0
Quantidade        19
Preco_Unitario    20
dtype: int64

In [5]:
# Preenche os valores ausentes com a média da coluna
pedidos_df["Quantidade"] = pedidos_df["Quantidade"].fillna(pedidos_df["Quantidade"].mean())
pedidos_df.head()

,ID_Pedido,Data,Cliente_ID,Item,Quantidade,Preco_Unitario
0,1,2023-11-16,222,Torta de Limao,23.000000,12.93
1,2,2024-05-25,232,Esfiha,15.902552,5.87
2,3,2024-12-20,276,Wrap de Frango,7.000000,25.98
3,4,2024-10-17,198,Batata Frita,15.902552,11.62
4,5,2023-07-06,385,Esfiha,17.000000,6.94


In [6]:
# Remove as linhas com valores ausentes
pedidos_df["Preco_Unitario"] = pedidos_df["Preco_Unitario"].dropna()
pedidos_df.head()

,ID_Pedido,Data,Cliente_ID,Item,Quantidade,Preco_Unitario
0,1,2023-11-16,222,Torta de Limao,23.000000,12.93
1,2,2024-05-25,232,Esfiha,15.902552,5.87
2,3,2024-12-20,276,Wrap de Frango,7.000000,25.98
3,4,2024-10-17,198,Batata Frita,15.902552,11.62
4,5,2023-07-06,385,Esfiha,17.000000,6.94


## Feature Engineering

In [7]:
# Cria coluna com o produto de quantidade e preço unitário
pedidos_df["Receita_Item"] = pedidos_df["Quantidade"] * pedidos_df["Preco_Unitario"]
pedidos_df.head()

,ID_Pedido,Data,Cliente_ID,Item,Quantidade,Preco_Unitario,Receita_Item
0,1,2023-11-16,222,Torta de Limao,23.000000,12.93,297.390000
1,2,2024-05-25,232,Esfiha,15.902552,5.87,93.347981
2,3,2024-12-20,276,Wrap de Frango,7.000000,25.98,181.860000
3,4,2024-10-17,198,Batata Frita,15.902552,11.62,184.787657
4,5,2023-07-06,385,Esfiha,17.000000,6.94,117.980000


## Agregações por Item

In [8]:
# Agrupa os pedidos por item e lista quantidade e receita total, ordenando do maior para o menor
(pedidos_df.groupby("Item")[["Quantidade", "Receita_Item"]].sum()
 .sort_values(
    by=["Quantidade", "Receita_Item"],
    ascending=False)
 .head())

,Quantidade,Receita_Item
Item,,
Hamburguer,561.805104,11562.230858
Brownie,471.902552,4603.498237
Cafe Expresso,440.902552,2657.710000
Acai 300ml,402.902552,6502.133503
Esfiha,399.707657,2379.656798


## Análise Temporal

In [9]:
# Converte a coluna data para o formato correto
pedidos_df["Data"] = pd.to_datetime(pedidos_df["Data"])

In [10]:
# Cria coluna contendo somente o mês do pedido
pedidos_df["Mes_Pedido"] = pedidos_df["Data"].dt.month
pedidos_df.head()

,ID_Pedido,Data,Cliente_ID,Item,Quantidade,Preco_Unitario,Receita_Item,Mes_Pedido
0,1,2023-11-16,222,Torta de Limao,23.000000,12.93,297.390000,11
1,2,2024-05-25,232,Esfiha,15.902552,5.87,93.347981,5
2,3,2024-12-20,276,Wrap de Frango,7.000000,25.98,181.860000,12
3,4,2024-10-17,198,Batata Frita,15.902552,11.62,184.787657,10
4,5,2023-07-06,385,Esfiha,17.000000,6.94,117.980000,7


In [11]:
# Agrupa pedidos por mês do pedido
pedidos_df.groupby("Mes_Pedido")["Receita_Item"].sum()

Mes_Pedido
1     10102.202668
2     16011.540000
3      7983.080000
4      8549.233503
5     12041.371067
6      9261.508863
7     10592.674084
8     12943.373503
9     14680.497517
10     5405.276984
11     6901.480000
12     8180.353341
Name: Receita_Item, dtype: float64

## Integração de Dados (Merge)

In [20]:
# Carrega dataframe do cardápio
cardapio_df = pd.read_csv('data/cardapio.csv')

In [13]:
# Cria um dataframe com o merge left de pedidos e cardápio
pedidos_cardapio_df = pd.merge(pedidos_df, cardapio_df, on="Item", how="left").head()

In [14]:
# Calcula a receita total por categoria
pedidos_cardapio_df.groupby("Categoria")["Receita_Item"].sum().sort_values(ascending=False)

Categoria
Salgados     396.115638
Doces        297.390000
Saudaveis    181.860000
Name: Receita_Item, dtype: float64

## Filtros e Consultas

In [15]:
# Aplica filtro de categoria e quantidade
filtro_categoria_quantidade_gt_10 = (pedidos_cardapio_df["Categoria"] == "Salgados") & (pedidos_cardapio_df["Quantidade"] > 10)
pedidos_cardapio_df[filtro_categoria_quantidade_gt_10].head()

,ID_Pedido,Data,Cliente_ID,Item,Quantidade,Preco_Unitario,Receita_Item,Mes_Pedido,Categoria,Preco_Base
1,2,2024-05-25,232,Esfiha,15.902552,5.87,93.347981,5,Salgados,6.5
3,4,2024-10-17,198,Batata Frita,15.902552,11.62,184.787657,10,Salgados,12.5
4,5,2023-07-06,385,Esfiha,17.000000,6.94,117.980000,7,Salgados,6.5


## KPIs e Análise Estatística com NumPy

In [16]:
# Receita Total por categoria
pedidos_cardapio_receita_total = pedidos_cardapio_df.groupby("Categoria")["Receita_Item"].sum()
pedidos_cardapio_receita_total.head()

Categoria
Doces        297.390000
Salgados     396.115638
Saudaveis    181.860000
Name: Receita_Item, dtype: float64

In [17]:
# Total de itens vendidos por categoria
pedidos_cardapio_quantidade_total = pedidos_cardapio_df.groupby("Categoria")["Quantidade"].sum()
pedidos_cardapio_quantidade_total.head()

Categoria
Doces        23.000000
Salgados     48.805104
Saudaveis     7.000000
Name: Quantidade, dtype: float64

In [22]:
resumo_categoria = pedidos_cardapio_df.groupby("Categoria")[["Receita_Item", "Quantidade"]].sum()
resumo_categoria["Ticket_Medio"] = resumo_categoria["Receita_Item"] / resumo_categoria["Quantidade"]
resumo_categoria

,Receita_Item,Quantidade,Ticket_Medio
Categoria,,,
Doces,297.390000,23.000000,12.930000
Salgados,396.115638,48.805104,8.116275
Saudaveis,181.860000,7.000000,25.980000


In [23]:
# Percentis (25%, 50%, 75%)
print("Preco Unitario:", np.percentile(pedidos_cardapio_df["Preco_Unitario"], [0.25, 0.5, 0.75]))
print("Quantidade:", np.percentile(pedidos_cardapio_df["Quantidade"], [0.25, 0.5, 0.75]))

Preco Unitario: [5.8807 5.8914 5.9021]
Quantidade: [7.08902552 7.17805104 7.26707657]
